# Airbnb Price Prediction — Chicago
A full ML pipeline from raw data to a deployable prediction function.

**Models:** Linear Regression, Random Forest  
**Target:** `price_log` (log of nightly price in USD)  
**Data:** Inside Airbnb — Chicago, September 2025

In [1]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import numpy as np
import folium
import matplotlib.cm as cm
import matplotlib.colors as colors
from sklearn.model_selection import StratifiedShuffleSplit
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression



## 1. Load Data
Read the raw CSV into a DataFrame. Nothing is modified here — this is the source of truth.

In [ ]:
def load_data(filepath: str) -> pd.DataFrame:
    """
    Load raw Airbnb listings CSV from disk.
    Returns a DataFrame with all original columns intact.
    """
    df = pd.read_csv(filepath)
    print(f"Loaded {len(df):,} rows, {df.shape[1]} columns")
    return df


filepath = os.path.join(os.getcwd(), "data/chicago-listings-sept-2025.csv")
df_raw = load_data(filepath)
df_raw.info()
df_raw.describe()

Loaded 8,663 rows, 18 columns
<class 'pandas.DataFrame'>
RangeIndex: 8663 entries, 0 to 8662
Data columns (total 18 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   id                              8663 non-null   int64  
 1   name                            8663 non-null   str    
 2   host_id                         8663 non-null   int64  
 3   host_name                       8042 non-null   str    
 4   neighbourhood_group             0 non-null      float64
 5   neighbourhood                   8663 non-null   str    
 6   latitude                        8663 non-null   float64
 7   longitude                       8663 non-null   float64
 8   room_type                       8663 non-null   str    
 9   price                           7828 non-null   float64
 10  minimum_nights                  8663 non-null   int64  
 11  number_of_reviews               8663 non-null   int64  
 12  last_review    

,id,host_id,neighbourhood_group,latitude,longitude,price,minimum_nights,number_of_reviews,reviews_per_month,calculated_host_listings_count,availability_365,number_of_reviews_ltm
count,8.663000e+03,8.663000e+03,0.0,8663.000000,8663.000000,7828.000000,8663.000000,8663.000000,6978.000000,8663.000000,8663.000000,8663.000000
mean,7.477230e+17,2.307973e+08,NaN,41.890671,-87.661372,601.126980,14.191620,56.846820,1.953534,55.075263,229.814845,13.816461
std,5.617424e+17,2.076739e+08,NaN,0.062124,0.043741,4199.822869,24.022535,96.182203,2.179434,155.051781,117.311943,21.717377
min,2.384000e+03,2.613000e+03,NaN,41.650640,-87.845893,13.000000,1.000000,0.000000,0.010000,1.000000,0.000000,0.000000
25%,4.873364e+07,4.693943e+07,NaN,41.858770,-87.685600,97.000000,2.000000,1.000000,0.540000,1.000000,141.000000,0.000000
50%,8.878846e+17,1.467262e+08,NaN,41.894796,-87.656800,153.000000,2.000000,18.000000,1.590000,4.000000,260.000000,6.000000
75%,1.254172e+18,4.310192e+08,NaN,41.933649,-87.629651,240.000000,32.000000,72.000000,2.890000,20.000000,338.000000,22.000000
max,1.514812e+18,7.185739e+08,NaN,42.021950,-87.528420,50032.000000,365.000000,1306.000000,69.100000,612.000000,365.000000,930.000000


In [5]:
df_raw.head()


,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365,number_of_reviews_ltm,license
0,2384,Hyde Park - Walk to University of Chicago,2613,Rebecca,NaN,Hyde Park,41.787900,-87.587800,Private room,119.0,3,257,2025-08-07,1.97,1,348,10,R17000015609
1,7126,Tiny Studio Apartment 94 Walk Score,17928,Sarah,NaN,West Town,41.901660,-87.680210,Entire home/apt,89.0,2,595,2025-09-01,3.01,1,302,47,R24000114046
2,10945,The Biddle House (#1),33004,At Home Inn,NaN,Lincoln Park,41.911960,-87.639810,Entire home/apt,203.0,4,129,2025-09-06,0.93,6,325,24,2209984
3,12140,Lincoln Park Guest House,46734,Shar And Robert,NaN,Lincoln Park,41.923570,-87.649470,Private room,339.0,2,19,2025-09-07,0.15,1,156,3,2398451
4,28749,Quirky Bucktown Loft w/ Parking No Parties,27506,Lauri,NaN,Logan Square,41.920226,-87.679613,Entire home/apt,258.0,2,265,2025-09-01,1.47,1,74,37,R24000113825


## 2. Clean Data
Drop columns we can't use, handle nulls, and remove price outliers.

In [6]:
def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Drop unusable columns, handle nulls, remove outliers, and log-transform price.
    Returns a cleaned DataFrame ready for exploration and feature engineering.
    """
    df = df.drop(columns=["id", "name", "host_id", "host_name", "license", "neighbourhood_group"])

    # drop rows with no price — can't train without a target
    df = df.dropna(subset=["price"]).reset_index(drop=True)

    # encode whether a listing has ever been reviewed
    df["has_been_reviewed"] = df["last_review"].notna().astype(int)

    # days since last review — 0 if never reviewed
    df["days_since_last_review"] = (
        pd.to_datetime("today") - pd.to_datetime(df["last_review"], errors="coerce")
    ).dt.days.fillna(0).astype(int)

    # fill missing reviews_per_month with 0 (listing has no reviews)
    df["reviews_per_month"] = df["reviews_per_month"].fillna(0)

    # remove top 1% price outliers
    upper = df["price"].quantile(0.9901)
    print("hi",upper)
    print(df[df["price"] > upper].sort_values("price", ascending=False))
    df = df[df["price"] <= upper]

    # log-transform price — reduces skew, improves model performance
    df["price_log"] = np.log1p(df["price"])

    df = df.drop(columns=["last_review"]).reset_index(drop=True)

    print(f"After cleaning: {len(df):,} rows")
    return df


df_clean = clean_data(df_raw)
df_clean.describe()

hi 3340.7846000000663
        neighbourhood   latitude  longitude        room_type    price  \
4336             Loop  41.884310 -87.633480       Hotel room  50032.0   
4907  Near North Side  41.896489 -87.623503     Private room  50000.0   
4674  Near North Side  41.893540 -87.632430       Hotel room  50000.0   
6773  Near North Side  41.894500 -87.631940       Hotel room  50000.0   
6772  Near North Side  41.894500 -87.631940       Hotel room  50000.0   
...               ...        ...        ...              ...      ...   
7289        West Town  41.902410 -87.672640     Private room   5400.0   
140         West Town  41.891100 -87.669470  Entire home/apt   5000.0   
1185        West Town  41.894990 -87.654780  Entire home/apt   4500.0   
3092     Lincoln Park  41.922520 -87.645360  Entire home/apt   4500.0   
4638             Loop  41.884624 -87.625857  Entire home/apt   3486.0   

      minimum_nights  number_of_reviews last_review  reviews_per_month  \
4336               1       

,latitude,longitude,price,minimum_nights,number_of_reviews,reviews_per_month,calculated_host_listings_count,availability_365,number_of_reviews_ltm,has_been_reviewed,days_since_last_review,price_log
count,7750.000000,7750.000000,7750.000000,7750.000000,7750.000000,7750.000000,7750.000000,7750.000000,7750.000000,7750.000000,7750.000000,7750.000000
mean,41.891045,-87.661706,204.968645,13.681935,59.624387,1.642227,58.284258,241.114452,14.556258,0.814839,324.574581,5.030715
std,0.062496,0.043314,213.258946,23.162660,97.799331,1.961216,160.049397,108.399164,19.369787,0.388453,326.068081,0.741759
min,41.650640,-87.845893,13.000000,1.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,2.639057
25%,41.858880,-87.686060,96.000000,2.000000,1.000000,0.130000,1.000000,157.000000,0.000000,1.000000,271.000000,4.574711
50%,41.895079,-87.657574,152.000000,2.000000,20.000000,1.190000,4.000000,265.000000,7.000000,1.000000,284.000000,5.030438
75%,41.934342,-87.630726,237.000000,32.000000,77.000000,2.637500,21.000000,340.000000,24.000000,1.000000,309.000000,5.472271
max,42.021950,-87.528420,3188.000000,365.000000,1267.000000,61.590000,612.000000,365.000000,388.000000,1.000000,4317.000000,8.067463


## 3. Explore Data
Visualize the cleaned data before engineering features — this is what motivates the decisions we make next.

Key questions:
- Where are the expensive listings concentrated geographically?
- How is price distributed across the dataset?
- Which raw features correlate most with price?

> **Note:** This step is for understanding only — skipped in production (`train.py`).

In [ ]:
def explore_data(df: pd.DataFrame) -> None:
    """
    Visualize price distribution, geographic spread, and feature correlations.
    Notebook-only — not called in train.py.

    Produces:
    - Price distribution histogram (log scale)
    - Scatter map of listings colored by price
    - Interactive folium map
    - Neighbourhood listing counts
    - Correlation heatmap of raw numeric features
    """
    # --- price distribution ---
    df["price"].hist(bins=50, figsize=(10, 4))
    plt.xlabel("price)")
    plt.ylabel("Count")
    plt.title("Distribution of Price")
    plt.tight_layout()
    plt.show()
        # --- price log distribution ---
    df["price_log"].hist(bins=50, figsize=(10, 4))
    plt.xlabel("log(price)")
    plt.ylabel("Count")
    plt.title("Distribution of log Price")
    plt.tight_layout()
    plt.show()

    # --- scatter map: size = review count, color = price ---
    df.plot(
        kind="scatter", x="longitude", y="latitude",
        alpha=0.4,
        s=df["number_of_reviews"] / 10,
        c="price_log",
        cmap="jet", colorbar=True, sharex=False,
        figsize=(10, 7),
        title="Listings by Location — size=reviews, color=price"
    )
    plt.show()

    # --- interactive folium map ---
    m = folium.Map(location=[41.8827, -87.6233], zoom_start=11)
    norm = colors.Normalize(vmin=df["price_log"].min(), vmax=df["price_log"].max())
    for _, row in df.iterrows():
        color = colors.to_hex(cm.jet(norm(row["price_log"])))
        folium.CircleMarker(
            location=[row["latitude"], row["longitude"]],
            radius=3,
            color=color,
            fill=True,
            fill_opacity=0.6,
            popup=f"${row['price']:.0f}"
        ).add_to(m)
    display(m)

    # --- neighbourhood listing counts ---
    plt.figure(figsize=(10, 20))
    df["neighbourhood"].value_counts().plot(kind="barh")
    plt.title("Listings per Neighbourhood")
    plt.tight_layout()
    plt.show()

    # --- correlation heatmap of raw numeric features ---
    # note: lat/lon are raw here — this is what motivates engineering dist_to_center
    corr = df.corr(numeric_only=True)
    mask = np.triu(np.ones_like(corr, dtype=bool))
    f, ax = plt.subplots(figsize=(11, 9))
    cmap = sns.diverging_palette(230, 20, as_cmap=True)
    sns.heatmap(
        corr, mask=mask, cmap=cmap, vmax=0.3, center=0,
        square=True, linewidths=5, cbar_kws={"shrink": 0.5}
    )
    plt.title("Feature Correlation Matrix (pre-engineering)")
    plt.show()


explore_data(df_clean)

## 4. Engineer Features
Create new features that better capture what drives price.
Now that we've seen the data, we can make motivated feature decisions.

Key decisions:
- `dist_to_center` — the map showed expensive listings cluster downtown; a single distance captures this better than raw lat/lon
- `min_nights_bin` — whether a listing targets nightly, weekly, or monthly stays matters more than the raw number
- Drop raw `latitude`, `longitude`, `minimum_nights` — replaced by engineered versions

In [ ]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add engineered features and drop columns they replace.
    Returns a DataFrame with new features ready for modelling.
    """
    chicago_center = (41.8781, -87.6298)

    # euclidean distance to city center — captures location value better than raw coordinates
    df["dist_to_center"] = np.sqrt(
        (df["latitude"] - chicago_center[0]) ** 2 +
        (df["longitude"] - chicago_center[1]) ** 2
    )

    # bin minimum nights into stay-type categories
    df["min_nights_bin"] = pd.cut(
        df["minimum_nights"],
        bins=[0, 1, 3, 7, 30, 365],
        labels=["nightly", "short", "weekly", "monthly", "longterm"]
    )

    # drop originals replaced by engineered features
    df = df.drop(columns=["latitude", "longitude", "minimum_nights"]).reset_index(drop=True)

    print(f"Features after engineering: {df.shape[1]} columns")
    return df


df_engineered = engineer_features(df_clean.copy())

## 5. Split Data
Split into train and test sets before touching features.

We use `StratifiedShuffleSplit` on binned `price_log` to ensure both sets have a representative price distribution — important with only 8K rows.

In [ ]:
def split_data(df: pd.DataFrame, test_size: float = 0.2, random_state: int = 42):
    """
    Stratified train/test split on price_log bins.
    Ensures price distribution is representative in both sets.
    Returns (train_set, test_set) as DataFrames with all columns.
    """
    df["price_log_cat"] = pd.cut(df["price_log"], bins=5, labels=False)

    split = StratifiedShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)

    for train_idx, test_idx in split.split(df, df["price_log_cat"]):
        train_set = df.loc[train_idx].drop(columns=["price_log_cat"])
        test_set = df.loc[test_idx].drop(columns=["price_log_cat"])

    print(f"Train: {len(train_set):,} rows | Test: {len(test_set):,} rows")
    return train_set, test_set


train_set, test_set = split_data(df_engineered)

# verify price distribution is similar between splits
print("\nTrain price_log mean:", train_set["price_log"].mean().round(4))
print("Test price_log mean: ", test_set["price_log"].mean().round(4))

## 6. Preprocess
Separate features from target, one-hot encode categoricals, and align columns.

Key decisions:
- One-hot encode `neighbourhood`, `room_type`, `min_nights_bin` — models need numeric inputs
- `drop_first=True` — avoids the dummy variable trap (perfect multicollinearity)
- `reindex` on val/test — ensures column alignment if a category appears in train but not val/test
- Encode on train only — prevents data leakage from val/test into the encoding

In [ ]:


from sklearn.preprocessing import OneHotEncoder


CATEGORICAL_COLS = ["neighbourhood", "room_type", "min_nights_bin"]


def preprocess(train_set: pd.DataFrame, test_set: pd.DataFrame):
    """
    Separate X/y, one-hot encode categoricals, and split train into train/val.
    Fit encoding on train only to prevent data leakage.
    Returns (X_train, X_val, X_test, y_train, y_val, y_test).
    """
    # separate features from target
    y_train_full = train_set["price_log"]
    X_train_full = train_set.drop(columns=train_set.filter(regex="^price").columns)

    y_test = test_set["price_log"]
    X_test = test_set.drop(columns=test_set.filter(regex="^price").columns)

    # split train into train/val
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full, y_train_full, test_size=0.2, random_state=42
    )

    # one-hot encode — fit on train, apply same columns to val and test
    encoder = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')
    encoder.fit(X_train[CATEGORICAL_COLS])
    def encode(df):
        encoded = encoder.transform(df[CATEGORICAL_COLS])
        encoded_df = pd.DataFrame(encoded, columns=encoder.get_feature_names_out(CATEGORICAL_COLS), index=df.index)
        return pd.concat([df.drop(columns=CATEGORICAL_COLS), encoded_df], axis=1)

    X_train = encode(X_train)
    X_val = encode(X_val)
    X_test = encode(X_test)
    # align columns — val/test may be missing categories that appear in train

    print(f"X_train shape: {X_train.shape}")
    print(f"X_val shape:   {X_val.shape}")
    print(f"X_test shape:  {X_test.shape}")

    return X_train, X_val, X_test, y_train, y_val, y_test


X_train, X_val, X_test, y_train, y_val, y_test = preprocess(train_set, test_set)

## 7. Train Models
Train Linear Regression and Random Forest on the training set.

In [ ]:
def train_models(X_train: pd.DataFrame, y_train: pd.Series)-> tuple[LinearRegression, RandomForestRegressor]:
    """
    Train Linear Regression and Random Forest on the training set.
    Returns (model_lr, model_rf).
    """
    model_lr = LinearRegression()
    model_lr.fit(X_train, y_train)
    print("Linear Regression trained")

    model_rf = RandomForestRegressor(n_estimators=100, random_state=42)
    model_rf.fit(X_train, y_train)
    print("Random Forest trained")

    return model_lr, model_rf


model_lr, model_rf = train_models(X_train, y_train)

## 8. Evaluate Models
Compare both models on train, val, and test sets.

In [ ]:
from sklearn.metrics import mean_squared_error


def evaluate_model(model, X_train, y_train, X_val, y_val, X_test, y_test, name: str):
    """
    Print R² on train/val/test and RMSE in both log and dollar scale.
    Dollar RMSE is more interpretable for a non-technical audience.
    """
    print(f"\n--- {name} ---")
    print(f"Train R²: {model.score(X_train, y_train):.4f}")
    print(f"Val R²:   {model.score(X_val, y_val):.4f}")
    print(f"Test R²:  {model.score(X_test, y_test):.4f}")

    y_pred_log = model.predict(X_test)
    rmse_log = np.sqrt(mean_squared_error(y_test, y_pred_log))
    print(f"Test RMSE (log scale): {rmse_log:.4f}")

    # convert back to dollars for interpretability
    rmse_dollars = np.sqrt(mean_squared_error(np.expm1(y_test), np.expm1(y_pred_log)))
    print(f"Test RMSE (dollars):   ${rmse_dollars:.2f}")


evaluate_model(model_lr, X_train, y_train, X_val, y_val, X_test, y_test, "Linear Regression")
evaluate_model(model_rf, X_train, y_train, X_val, y_val, X_test, y_test, "Random Forest")

## 9. Predict Price
The function the API will call — takes a single listing's details and returns predicted price in dollars from both models.

In [ ]:
def predict_price(model_lr, model_rf, listing_input: dict, train_columns: list) -> dict:
    """
    Predict nightly price in dollars from a listing's raw features.

    Args:
        model_lr: trained LinearRegression model
        model_rf: trained RandomForestRegressor model
        listing_input: dict of raw feature values (neighbourhood, room_type, lat, lon, etc.)
        train_columns: column list from X_train — ensures input matches what the model expects

    Returns:
        dict with 'linear_regression' and 'random_forest' predictions in USD
    """
    listing = listing_input.copy()

    # replicate feature engineering from engineer_features()
    chicago_center = (41.8781, -87.6298)
    listing["dist_to_center"] = np.sqrt(
        (listing.pop("latitude", 0) - chicago_center[0]) ** 2 +
        (listing.pop("longitude", 0) - chicago_center[1]) ** 2
    )

    min_nights = listing.pop("minimum_nights", 1)
    if min_nights <= 1:
        listing["min_nights_bin"] = "nightly"
    elif min_nights <= 3:
        listing["min_nights_bin"] = "short"
    elif min_nights <= 7:
        listing["min_nights_bin"] = "weekly"
    elif min_nights <= 30:
        listing["min_nights_bin"] = "monthly"
    else:
        listing["min_nights_bin"] = "longterm"

    # encode and align to training columns
    input_df = pd.DataFrame([listing])
    input_df = pd.get_dummies(input_df, columns=CATEGORICAL_COLS, drop_first=True)
    input_df = input_df.reindex(columns=train_columns, fill_value=0)

    lr_pred = np.expm1(model_lr.predict(input_df)[0])
    rf_pred = np.expm1(model_rf.predict(input_df)[0])

    return {
        "linear_regression": round(lr_pred, 2),
        "random_forest": round(rf_pred, 2)
    }


# example — what would a 2-night entire apartment in the Loop cost?
example_listing = {
    "neighbourhood": "Loop",
    "latitude": 41.8827,
    "longitude": -87.6233,
    "room_type": "Entire home/apt",
    "minimum_nights": 2,
    "number_of_reviews": 10,
    "reviews_per_month": 1.5,
    "calculated_host_listings_count": 1,
    "availability_365": 200,
    "number_of_reviews_ltm": 5,
    "has_been_reviewed": 1,
    "days_since_last_review": 30,
}

prediction = predict_price(model_lr, model_rf, example_listing, X_train.columns.tolist())
print(f"Linear Regression prediction: ${prediction['linear_regression']}")
print(f"Random Forest prediction:     ${prediction['random_forest']}")

In [ ]:
pwd = os.getcwd()
filepath = os.path.join(pwd, "chicago-listings-sept-2025.csv")
filepath

In [ ]:
airbnb_listings = pd.read_csv(filepath)
airbnb_listings.info()
airbnb_listings.describe()
## things to drop care of
## id 
## name
## host name
## neighborhood group
## license

In [ ]:
airbnb_listings.nlargest(10, 'price')


In [ ]:
airbnb_listings_dropped= airbnb_listings.drop(columns=['id','name','host_id', 'host_name' ,'license','neighbourhood_group'])
airbnb_listings_dropped= airbnb_listings_dropped.dropna(subset=['price']).reset_index(drop=True)
airbnb_listings_dropped['has_been_reviewed'] = airbnb_listings_dropped['last_review'].notna().astype(int)
airbnb_listings_dropped['days_since_last_review']=(pd.to_datetime('today')- pd.to_datetime(airbnb_listings_dropped['last_review'], errors='coerce')).dt.days.fillna(0).astype(int)
airbnb_listings_dropped['reviews_per_month'] = airbnb_listings_dropped['reviews_per_month'].fillna(0)
upper = airbnb_listings_dropped['price'].quantile(0.99)
airbnb_listings_clean = airbnb_listings_dropped[airbnb_listings_dropped['price'] <= upper]
airbnb_listings_clean["price_log"]= np.log1p(airbnb_listings_clean['price'])
airbnb_listings_clean=airbnb_listings_clean.drop(columns=['last_review']).reset_index(drop=True)
plt.figure(figsize=(10, 20))  # (width, height)
airbnb_listings_clean["neighbourhood"].value_counts().plot(kind="barh")
plt.tight_layout()
plt.show()


In [ ]:
airbnb_listings_clean.describe()

In [ ]:
airbnb_listings_clean["price_log"].hist(bins=5, figsize=(10,5))
plt.xlabel("Price (USD)")
plt.ylabel("Count")
plt.title("Distribution of Airbnb Prices")
plt.show()

In [ ]:
airbnb_listings_clean.plot(kind="scatter", x="longitude", y="latitude",
    alpha=0.4,
    s=airbnb_listings_clean["number_of_reviews"]/10, 
    c="price_log",                   
    cmap="jet", colorbar=True, sharex=False,
    figsize=(10,7))

In [ ]:
m = folium.Map(location=[41.8827, -87.6233], zoom_start=11)

norm = colors.Normalize(vmin=airbnb_listings_clean['price_log'].min(), 
                        vmax=airbnb_listings_clean['price_log'].max())

for _, row in airbnb_listings_clean.iterrows():
    color = colors.to_hex(cm.jet(norm(row['price_log'])))
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=3,
        color=color,
        fill=True,
        fill_opacity=0.6,
        popup=f"${row['price']:.0f}"
    ).add_to(m)

m

In [ ]:
import seaborn as sns
# cols_to_drop = [col for col in housing.columns if col.startswith("property_type_") or col.startswith("places_")]
# housing_corr = housing.drop(columns=cols_to_drop)
chicago_center = (41.8781, -87.6298)
airbnb_listings_clean["dist_to_center"] = np.sqrt(
    (airbnb_listings_clean["latitude"] - chicago_center[0])**2 +
    (airbnb_listings_clean["longitude"] - chicago_center[1])**2
)
airbnb_listings_clean["min_nights_bin"] = pd.cut(
    airbnb_listings_clean["minimum_nights"],
    bins=[0, 1, 3, 7, 30, 365],
    labels=["nightly", "short", "weekly", "monthly", "longterm"]
)
airbnb_listings_clean= airbnb_listings_clean.drop(columns=["latitude", "longitude"]).reset_index(drop=True)
corr= airbnb_listings_clean.corr(numeric_only=True)

mask = np.triu(np.ones_like(corr, dtype=bool))

f,ax= plt.subplots(figsize=(11,9))
cmap= sns.diverging_palette(230,20, as_cmap=True)
sns.heatmap(corr, mask=mask, cmap=cmap, vmax=0.3, center=0, square=True, linewidths=5, cbar_kws={"shrink":0.5})

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit, train_test_split

airbnb_listings_clean["price_log_cat"] = pd.cut(
    airbnb_listings_clean["price_log"], bins=5, labels=False
)

split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

for train_index, test_index in split.split(
    airbnb_listings_clean, airbnb_listings_clean["price_log_cat"]
):
    strat_train_set = airbnb_listings_clean.loc[train_index].drop(
        columns=["price_log_cat"]
    )
    strat_test_set = airbnb_listings_clean.loc[test_index].drop(
        columns=["price_log_cat"]
    )


y = strat_train_set["price_log"]
X = strat_train_set.drop(columns=strat_train_set.filter(regex="^price").columns)

# 3. Further split train into train/validation
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
X_train = pd.get_dummies(
    X_train, 
    columns=["neighbourhood", "room_type", "min_nights_bin"], 
    drop_first=True
)
X_val = pd.get_dummies(
    X_val, 
    columns=["neighbourhood", "room_type", "min_nights_bin"], 
    drop_first=True
)
X_val = X_val.reindex(columns=X_train.columns, fill_value=0)
X_val = X_val.reindex(columns=X_train.columns, fill_value=0)
X.dtypes
X_train.isnull().sum()


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

model = LinearRegression()
model.fit(X_train, y_train)
print("Train R²:", model.score(X_train, y_train))
print("Val R²:", model.score(X_val, y_val))

y_pred = model.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, y_pred))
print("Val RMSE:", rmse)

In [ ]:
from sklearn.linear_model import Ridge, Lasso

model_ridge = Ridge(alpha=1.0)
model_ridge.fit(X_train, y_train)
print("Ridge Train R²:", model_ridge.score(X_train, y_train))
print("Ridge Val R²:", model_ridge.score(X_val, y_val))

model_lasso = Lasso(alpha=0.01)
model_lasso.fit(X_train, y_train)
print("Lasso Train R²:", model_lasso.score(X_train, y_train))
print("Lasso Val R²:", model_lasso.score(X_val, y_val))

In [ ]:
from sklearn.ensemble import RandomForestRegressor

model_rf = RandomForestRegressor(n_estimators=100, random_state=42)
model_rf.fit(X_train, y_train)
y_test = strat_test_set["price_log"]
X_test = strat_test_set.drop(columns=strat_test_set.filter(regex="^price").columns)

X_test = pd.get_dummies(X_test, columns=["neighbourhood", "room_type"], drop_first=True)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

print("Test R²:", model_rf.score(X_test, y_test))
print("Train R²:", model_rf.score(X_train, y_train))
print("Val R²:", model_rf.score(X_val, y_val))

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

model_gb = GradientBoostingRegressor(n_estimators=100, random_state=42)
model_gb.fit(X_train, y_train)
y_test = strat_test_set["price_log"]
X_test = strat_test_set.drop(columns=strat_test_set.filter(regex="^price").columns)

X_test = pd.get_dummies(X_test, columns=["neighbourhood", "room_type"], drop_first=True)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

print("Test R²:", model_gb.score(X_test, y_test))
print("Train R²:", model_gb.score(X_train, y_train))
print("Val R²:", model_gb.score(X_val, y_val))

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [10, 20, 30, None],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"]
}

search = RandomizedSearchCV(
    RandomForestRegressor(random_state=42),
    param_grid,
    n_iter=20,
    cv=5,
    scoring="r2",
    random_state=42
)
search.fit(X_train, y_train)

print("Best params:", search.best_params_)
print("Val R²:", search.best_estimator_.score(X_val, y_val))